# Ejemplo de uso del modulo con dataset EOD

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "EOD_STGO" 

In [2]:
import pandas as pd
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

In [96]:
from __future__ import annotations

from io import StringIO
import re
from typing import Optional

import numpy as np
import pandas as pd
from pyproj import Transformer

import h3


# -----------------------------------------------------------------------------
# Configuración básica
# -----------------------------------------------------------------------------

# En mi código viejo EOD venía en EPSG:5361 y lo pasaba a WGS84.
# Lo dejo parametrizable, pero este sigue siendo el default recomendado.
DEFAULT_SOURCE_CRS = "EPSG:5361"
DEFAULT_TARGET_CRS = "EPSG:4326"

TRIP_FACTOR_COLS = [
    "FactorLaboralNormal",
    "FactorSabadoNormal",
    "FactorDomingoNormal",
    "FactorLaboralEstival",
    "FactorFindesemanaEstival",
]

TRIP_LOOKUP_TABLES = {
    "ComunaOrigen": "Comuna.csv",
    "ComunaDestino": "Comuna.csv",
    "SectorOrigen": "Sector.csv",
    "SectorDestino": "Sector.csv",
    "Proposito": "Proposito.csv",
    "PropositoAgregado": "PropositoAgregado.csv",
    "ActividadDestino": "ActividadDestino.csv",
    "ModoAgregado": "ModoAgregado.csv",
    "ModoPriPub": "ModoPriPub.csv",
    "ModoMotor": "ModoMotor.csv",
    "Periodo": "Periodo.csv",
    "TiempoMedio": "TiempoMedio.csv",
    "CodigoTiempo": "CodigoTiempo.csv",
}

PERSON_LOOKUP_TABLES = {
    "Sexo": "Sexo.csv",
    "Relacion": "Relacion.csv",
    "LicenciaConducir": "LicenciaConducir.csv",
    "PaseEscolar": "PaseEscolar.csv",
    "AdultoMayor": "AdultoMayor.csv",
    "Estudios": "Estudios.csv",
    "Actividad": "Actividad.csv",
    "Ocupacion": "Ocupacion.csv",
    "ActividadEmpresa": "ActividadEmpresa.csv",
    "JornadaTrabajo": "JornadaTrabajo.csv",
    "DondeEstudia": "Donde Estudia.csv",
    "MedioViajeRestricion": "MedioViajeRestriccion.csv",
    "ConoceTransantiago": "ConoceSantiago.csv",
    "NoUsaTransantiago": "NoUsaTransantiago.csv",
    "Discapacidad": "Discapacidad.csv",
    "TieneIngresos": "TieneIngresos.csv",
    "TramoIngreso": "TramoIngreso.csv",
    "TramoIngresoFinal": "TramoIngreso.csv",
    "IngresoImputado": "IngresoImputado.csv",
}

HOUSEHOLD_LOOKUP_TABLES = {
    "TipoDia": "TipoDia.csv",
    "Temporada": "Temporada.csv",
    "Propiedad": "Propiedad.csv",
}

STAGE_LOOKUP_TABLES = {
    "Autopista": "Autopista.csv",
    "ComunaOrigen": "Comuna.csv",
    "ComunaDestino": "Comuna.csv",
    "SectorOrigen": "Sector.csv",
    "SectorDestino": "Sector.csv",
    "Modo": "Modo.csv",
    "FormaPago": "Formapago.csv",
    "Estaciona": "Estaciona.csv",
    "RecorridoTransantiago": "RecorridoTransantiago.csv",
    "EstacionTrenIni": "EstacionTren.csv",
    "EstacionTrenFin": "EstacionTren.csv",
    "EstacionMetroIni": "EstacionMetro.csv",
    "EstacionMetroFin": "EstacionMetro.csv",
    "HorarioMetro": "HorarioMetro.csv",
    "EstacionMetroCambio": "EstacionMetroCambio.csv",
    "PropiedadBicicleta": "PropiedadBicicleta.csv",
    "UsaCiclovia": "UsaCiclovia.csv",
    "CirculacionBicicleta": "CirculacionBicicleta.csv",
    "EstacionaBicicleta": "EstacionaBicicleta.csv",
    "ModoEstacionaBicicleta": "ModoestacionaBicicleta.csv",
    "UsoHabitualBicicleta": "UsoHabitualBicicleta.csv",
}

# Algunas columnas de viajes resumidos son útiles para traer contexto a etapas.
# Al final esta variable/lista no se uso
TRIP_CONTEXT_COLS_FOR_STAGES = [
    "Etapas",
    "Proposito",
    "PropositoAgregado",
    "ActividadDestino",
    "ModoAgregado",
    "HoraIni",
    "HoraFin",
    "TiempoViaje",
    "TiempoMedio",
    "Periodo",
    *TRIP_FACTOR_COLS,
]

PERSON_COLS_USEFUL_FOR_TRIPS = [
    "Hogar",
    "Persona",
    "AnoNac",
    "Sexo",
    "Actividad",
    "Ocupacion",
    "LicenciaConducir",
    "PaseEscolar",
    "AdultoMayor",
    "TieneIngresos",
    "TramoIngreso",
]

HOUSEHOLD_COLS_USEFUL_FOR_TRIPS = [
    "Hogar",
    "Fecha",
    "TipoDia",
    "Temporada",
    "NumVeh",
    "NumBicAdulto",
    "NumBicNino",
    "IngresoHogar",
    "Comuna",
]

PERSON_COLS_USEFUL_FOR_STAGES = [
    "Hogar",
    "Persona",
    "AnoNac",
    "Sexo",
    "Actividad",
    "Ocupacion",
    "LicenciaConducir",
    "PaseEscolar",
    "AdultoMayor",
    "TieneIngresos",
    "TramoIngreso",
]

HOUSEHOLD_COLS_USEFUL_FOR_STAGES = [
    "Hogar",
    "Fecha",
    "TipoDia",
    "Temporada",
    "NumVeh",
    "NumBicAdulto",
    "NumBicNino",
    "IngresoHogar",
    "Comuna",
]

TRIP_COLS_USEFUL_FOR_STAGES = [
    "Hogar",
    "Persona",
    "Viaje",
    "Etapas",
    "Proposito",
    "PropositoAgregado",
    "ActividadDestino",
    "ModoAgregado",
    "HoraIni",
    "HoraFin",
    "TiempoViaje",
    "TiempoMedio",
    "Periodo",
    "FactorLaboralNormal",
    "FactorSabadoNormal",
    "FactorDomingoNormal",
    "FactorLaboralEstival",
    "FactorFindesemanaEstival",
]

# -----------------------------------------------------------------------------
# Lectura de tablas auxiliares reales desde directorio
# -----------------------------------------------------------------------------

def _normalize_aux_filename(filename: str) -> str:
    """
    Normaliza nombres de archivos auxiliares para usarlos como llave del dict.
    """
    return str(filename).strip()


def _read_aux_csv(csv_path: Path) -> pd.DataFrame:
    """
    Lee un CSV auxiliar EOD probando combinaciones razonables de separador y encoding.
    """
    attempts = [
        {"sep": ";", "encoding": "utf-8", "engine": "c"},
        {"sep": ",", "encoding": "utf-8", "engine": "c"},
        {"sep": ";", "encoding": "latin-1", "engine": "c"},
        {"sep": ",", "encoding": "latin-1", "engine": "c"},
        {"sep": ";", "encoding": "cp1252", "engine": "c"},
        {"sep": ",", "encoding": "cp1252", "engine": "c"},
        # fallback final más tolerante
        {"sep": None, "encoding": "latin-1", "engine": "python"},
    ]

    last_error = None

    for cfg in attempts:
        try:
            kwargs = dict(
                filepath_or_buffer=csv_path,
                dtype=str,
            )

            if cfg["engine"] == "python":
                kwargs.update(
                    sep=cfg["sep"],
                    encoding=cfg["encoding"],
                    engine="python",
                )
            else:
                kwargs.update(
                    sep=cfg["sep"],
                    encoding=cfg["encoding"],
                    engine="c",
                    low_memory=False,
                )

            df = pd.read_csv(**kwargs)

            # Si el parseo salió mal y quedó una sola columna enorme, seguimos probando
            if df.shape[1] <= 1:
                continue

            return df

        except Exception as e:
            last_error = e
            continue

    raise RuntimeError(f"No se pudo leer la tabla auxiliar {csv_path.name}: {last_error}")


def load_eod_aux_tables_from_dir(aux_dir: str | Path) -> dict[str, pd.DataFrame]:
    """
    Carga todas las tablas auxiliares EOD desde un directorio real con CSVs.

    Espera una estructura tipo:
        Tablas_parametros/
            Actividad.csv
            ActividadDestino.csv
            ...
    Devuelve:
        dict[nombre_archivo_csv -> DataFrame]
    """
    aux_dir = Path(aux_dir)

    if not aux_dir.exists():
        raise FileNotFoundError(f"No existe el directorio de tablas auxiliares: {aux_dir}")

    if not aux_dir.is_dir():
        raise NotADirectoryError(f"La ruta de tablas auxiliares no es un directorio: {aux_dir}")

    tables: dict[str, pd.DataFrame] = {}

    for csv_path in sorted(aux_dir.glob("*.csv")):
        df = _read_aux_csv(csv_path)

        df.columns = [
            str(c).strip().replace("Código", "Codigo")
            for c in df.columns
        ]

        tables[_normalize_aux_filename(csv_path.name)] = df

    return tables


def _normalize_lookup_key(value) -> Optional[str]:
    """
    Normaliza llaves que vienen como string, enteros, '1.0', etc.
    """
    if pd.isna(value):
        return None

    s = str(value).strip().strip('"')
    if not s:
        return None

    if s.endswith(".0"):
        s = s[:-2]

    return s


def build_lookup(
    aux_tables: dict[str, pd.DataFrame],
    table_name: str,
) -> dict[str, str]:
    """
    Construye un dict clave -> label usando una tabla auxiliar.

    Regla:
    - usa la primera columna como clave principal;
    - usa la segunda columna como valor/label;
    - si existe una tercera columna auxiliar (por ejemplo 'Campo2'),
      también la incorpora como clave alternativa hacia el mismo label.

    Esto permite soportar tablas donde la fuente usa:
    - códigos numéricos/ID, o
    - códigos alfabéticos auxiliares.
    """
    ref = aux_tables[table_name].copy()

    if ref.shape[1] < 2:
        raise ValueError(f"La tabla auxiliar {table_name} no tiene al menos 2 columnas.")

    key_col = ref.columns[0]
    value_col = ref.columns[1]
    alt_key_cols = list(ref.columns[2:])

    lookup: dict[str, str] = {}

    for _, row in ref.iterrows():
        label = row[value_col]
        if pd.isna(label):
            continue

        label = str(label).strip()

        # clave principal
        main_key = _normalize_lookup_key(row[key_col])
        if main_key is not None:
            lookup[main_key] = label

        # claves alternativas (ej. Campo2 = A/B/C)
        for alt_col in alt_key_cols:
            alt_key = _normalize_lookup_key(row[alt_col])
            if alt_key is not None:
                lookup[alt_key] = label

    return lookup


def decode_column_with_lookup_inplace(
    df: pd.DataFrame,
    column: str,
    lookup: dict[str, str],
    *,
    keep_original: bool = False,
) -> None:
    """
    Reemplaza en la misma columna los códigos por sus valores textuales.
    Si keep_original=True, guarda la original como '<col>_raw'.
    """
    if column not in df.columns:
        return

    if keep_original:
        df[f"{column}_raw"] = df[column]

    decoded = df[column].map(lambda x: lookup.get(_normalize_lookup_key(x), pd.NA))
    df[column] = decoded.where(decoded.notna(), df[column])


# -----------------------------------------------------------------------------
# Helpers numéricos / coords / tiempos
# -----------------------------------------------------------------------------

def parse_number_series(series: pd.Series) -> pd.Series:
    """
    Convierte strings tipo '338975,75' a float.
    """
    return pd.to_numeric(
        series.astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )


def add_wgs84_from_xy(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    out_lon: str,
    out_lat: str,
    source_crs: str = DEFAULT_SOURCE_CRS,
    target_crs: str = DEFAULT_TARGET_CRS,
) -> None:
    """
    Convierte coordenadas proyectadas XY a WGS84.
    Filtra internamente filas inválidas, pero no las elimina.
    """
    x = parse_number_series(df[x_col])
    y = parse_number_series(df[y_col])

    valid = x.notna() & y.notna() & (x != 0) & (y != 0)

    lon = pd.Series(np.nan, index=df.index, dtype="float64")
    lat = pd.Series(np.nan, index=df.index, dtype="float64")

    if valid.any():
        transformer = Transformer.from_crs(source_crs, target_crs, always_xy=True)
        valid_x = x.loc[valid].to_numpy()
        valid_y = y.loc[valid].to_numpy()
        lon_vals, lat_vals = transformer.transform(valid_x, valid_y)

        lon.loc[valid] = lon_vals
        lat.loc[valid] = lat_vals

    df[out_lon] = lon
    df[out_lat] = lat
    df.drop(columns=[x_col], inplace=True)
    df.drop(columns=[y_col], inplace=True)



def normalize_hhmm(value) -> Optional[str]:
    """
    Normaliza horarios tipo '5:10' -> '05:10'.
    Si no cumple HH:MM, devuelve NA.
    """
    if pd.isna(value):
        return pd.NA

    s = str(value).strip()
    m = re.match(r"^(\d{1,2}):(\d{1,2})$", s)
    if not m:
        return pd.NA

    hh = int(m.group(1))
    mm = int(m.group(2))

    if 0 <= hh <= 23 and 0 <= mm <= 59:
        return f"{hh:02d}:{mm:02d}"

    return pd.NA


def first_non_null_numeric(df: pd.DataFrame, columns: list[str]) -> pd.Series:
    """
    Toma la primera columna numérica no nula, en orden.
    """
    out = pd.Series(np.nan, index=df.index, dtype="float64")

    for col in columns:
        if col not in df.columns:
            continue
        vals = parse_number_series(df[col])
        out = out.where(out.notna(), vals)

    return out

def decode_list_column_with_lookup_inplace(
    df: pd.DataFrame,
    column: str,
    lookup: dict[str, str],
    *,
    sep: str = ";",
    keep_original: bool = False,
) -> None:
    """
    Reemplaza una columna cuyo contenido es una lista de IDs separadas por `sep`
    por la misma lista, pero con valores decodificados.
    """
    if column not in df.columns:
        return

    if keep_original:
        df[f"{column}_raw"] = df[column]

    def _decode_value(value):
        if pd.isna(value):
            return pd.NA

        raw = str(value).strip().strip('"')
        if not raw:
            return pd.NA

        tokens = [t.strip() for t in raw.split(sep) if t.strip()]
        decoded_tokens = []

        for tok in tokens:
            nk = _normalize_lookup_key(tok)
            if nk is None:
                continue
            decoded_tokens.append(lookup.get(nk, tok))

        if not decoded_tokens:
            return pd.NA

        return sep.join(decoded_tokens)

    df[column] = df[column].map(_decode_value)

# -----------------------------------------------------------------------------
# Caso 1: viajes resumidos EOD -> dataframe listo para import
# -----------------------------------------------------------------------------

def prepare_eod_triplevel_for_import(
    df_viajes: pd.DataFrame,
    df_personas: Optional[pd.DataFrame],
    df_hogares: Optional[pd.DataFrame],
    aux_dir: str | Path,
    *,
    source_crs: str = DEFAULT_SOURCE_CRS,
    target_crs: str = DEFAULT_TARGET_CRS,
    drop_rows_without_minimal_od: bool = False,
) -> pd.DataFrame:
    """
    Preprocess manual nivel 1 para EOD viajes resumidos.

    Qué hace:
    - conserva todas las columnas originales;
    - agrega columnas decodificadas *_label;
    - mergea personas y hogares;
    - crea columnas puente con nombres Golondrina;
    - deja un dataframe listo para importar como viajes resumidos Tier 2.
    """
    aux_tables = load_eod_aux_tables_from_dir(aux_dir)
    required_lookup_files = set(TRIP_LOOKUP_TABLES.values()) | set(PERSON_LOOKUP_TABLES.values()) | set(HOUSEHOLD_LOOKUP_TABLES.values())
    missing = [name for name in required_lookup_files if name not in aux_tables]
    if missing:
        print("Advertencia: faltan tablas auxiliares:", missing)

    needed_tables = (
        set(TRIP_LOOKUP_TABLES.values())
        | set(PERSON_LOOKUP_TABLES.values())
        | set(HOUSEHOLD_LOOKUP_TABLES.values())
        | {"Modo.csv"}
    )
    lookups = {name: build_lookup(aux_tables, name) for name in needed_tables}

    df = df_viajes.copy()

    # 1) Enriquecimiento por persona y hogar.
    if df_personas is not None:
        cols = [c for c in PERSON_COLS_USEFUL_FOR_TRIPS if c in df_personas.columns]
        df = df.merge(
            df_personas[cols].copy(),
            on=["Hogar", "Persona"],
            how="left",
            suffixes=("", "_persona"),
        )

    if df_hogares is not None:
        cols = [c for c in HOUSEHOLD_COLS_USEFUL_FOR_TRIPS if c in df_hogares.columns]
        df = df.merge(
            df_hogares[cols].copy(),
            on=["Hogar"],
            how="left",
            suffixes=("", "_hogar"),
        )

    # 2) Decodificación de IDs/códigos de la tabla de viajes.
    for col, table_name in TRIP_LOOKUP_TABLES.items():
        decode_column_with_lookup_inplace(df, col, lookups[table_name], keep_original=False)

    # 3) Decodificación de atributos de personas.
    for col, table_name in PERSON_LOOKUP_TABLES.items():
        decode_column_with_lookup_inplace(df, col, lookups[table_name], keep_original=False)

    # 4) Decodificación de atributos de hogares.
    for col, table_name in HOUSEHOLD_LOOKUP_TABLES.items():
        decode_column_with_lookup_inplace(df, col, lookups[table_name], keep_original=False)

    # 5) Decodificación de MediosUsados -> secuencia textual cruda.
    if "MediosUsados" in df.columns:
        decode_list_column_with_lookup_inplace(
            df,
            "MediosUsados",
            lookups["Modo.csv"],
            sep=";",
            keep_original=False,
        )

        # 6) Construcción de coordenadas WGS84 + H3, pero en columnas "fuente-like"
    add_wgs84_from_xy(
        df,
        x_col="OrigenCoordX",
        y_col="OrigenCoordY",
        out_lon="OrigenCoordLon",
        out_lat="OrigenCoordLat",
        source_crs=source_crs,
        target_crs=target_crs,
    )
    add_wgs84_from_xy(
        df,
        x_col="DestinoCoordX",
        y_col="DestinoCoordY",
        out_lon="DestinoCoordLon",
        out_lat="DestinoCoordLat",
        source_crs=source_crs,
        target_crs=target_crs,
    )

    # Peso expandido útil, pero con nombre no canónico.
    df["factor_expansion"] = first_non_null_numeric(df, TRIP_FACTOR_COLS)

    if "Etapas" in df.columns:
        df["cantidad_etapas"] = pd.to_numeric(df["Etapas"], errors="coerce").astype("Int64")

    return df


# -----------------------------------------------------------------------------
# Caso 2: etapas EOD -> dataframe listo para import
# -----------------------------------------------------------------------------

def prepare_eod_stagelevel_for_import(
    df_etapas: pd.DataFrame,
    df_viajes: Optional[pd.DataFrame],
    df_personas: Optional[pd.DataFrame],
    df_hogares: Optional[pd.DataFrame],
    aux_dir: str | Path,
    *,
    source_crs: str = DEFAULT_SOURCE_CRS,
    target_crs: str = DEFAULT_TARGET_CRS,
    drop_rows_without_minimal_od: bool = False,
) -> pd.DataFrame:
    """
    Preprocess manual nivel 1 para EOD etapas.

    Qué hace:
    - conserva todas las columnas originales de etapas;
    - agrega columnas decodificadas *_label;
    - mergea personas y hogares;
    - opcionalmente trae contexto de viajes resumidos;
    - crea columnas puente con nombres Golondrina;
    - deja un dataframe listo para importar como movements/stages,
      naturalmente en Tier 3 (porque la tabla de etapas no trae horas por etapa).
    """
    aux_tables = load_eod_aux_tables_from_dir(aux_dir)
    required_lookup_files = set(STAGE_LOOKUP_TABLES.values()) | set(PERSON_LOOKUP_TABLES.values()) | set(HOUSEHOLD_LOOKUP_TABLES.values())
    missing = [name for name in required_lookup_files if name not in aux_tables]
    if missing:
        print("Advertencia: faltan tablas auxiliares:", missing)

    needed_tables = (
        set(STAGE_LOOKUP_TABLES.values())
        | set(PERSON_LOOKUP_TABLES.values())
        | set(HOUSEHOLD_LOOKUP_TABLES.values())
        | {"Proposito.csv", "PropositoAgregado.csv", "ActividadDestino.csv", "ModoAgregado.csv", "Periodo.csv", "TiempoMedio.csv", "Autopista.csv"}
    )
    lookups = {name: build_lookup(aux_tables, name) for name in needed_tables}

    df = df_etapas.copy()

    if "Autopistas" in df.columns:
        decode_list_column_with_lookup_inplace(
            df,
            "Autopistas",
            lookups["Autopista.csv"],
            sep=";",
            keep_original=False,
        )

    # 1) Traigo contexto de viajes si está disponible.
    if df_personas is not None:
        cols = [c for c in PERSON_COLS_USEFUL_FOR_STAGES if c in df_personas.columns]
        df = df.merge(
            df_personas[cols].copy(),
            on=["Hogar", "Persona"],
            how="left",
            suffixes=("", "_persona"),
        )

    if df_hogares is not None:
        cols = [c for c in HOUSEHOLD_COLS_USEFUL_FOR_STAGES if c in df_hogares.columns]
        df = df.merge(
            df_hogares[cols].copy(),
            on=["Hogar"],
            how="left",
            suffixes=("", "_hogar"),
        )

    if df_viajes is not None:
        cols = [c for c in TRIP_COLS_USEFUL_FOR_STAGES if c in df_viajes.columns]
        trip_ctx = df_viajes[cols].copy()

        rename_map = {
            c: f"trip_{c}"
            for c in cols
            if c not in {"Hogar", "Persona", "Viaje"}
        }
        trip_ctx = trip_ctx.rename(columns=rename_map)

        df = df.merge(
            trip_ctx,
            on=["Hogar", "Persona", "Viaje"],
            how="left",
        )

    # 3) Decodificación de columnas propias de etapas.
    for col, table_name in STAGE_LOOKUP_TABLES.items():
        decode_column_with_lookup_inplace(df, col, lookups[table_name], keep_original=False)

    # 4) Decodificación de personas.
    for col, table_name in PERSON_LOOKUP_TABLES.items():
        decode_column_with_lookup_inplace(df, col, lookups[table_name], keep_original=False)

    # 5) Decodificación de hogares.
    for col, table_name in HOUSEHOLD_LOOKUP_TABLES.items():
        decode_column_with_lookup_inplace(df, col, lookups[table_name], keep_original=False)

    # 6) Decodificación del contexto de viaje propagado a etapas.
    trip_context_lookup_map = {
        "trip_Proposito": "Proposito.csv",
        "trip_PropositoAgregado": "PropositoAgregado.csv",
        "trip_ActividadDestino": "ActividadDestino.csv",
        "trip_ModoAgregado": "ModoAgregado.csv",
        "trip_Periodo": "Periodo.csv",
        "trip_TiempoMedio": "TiempoMedio.csv",
    }
    for col, table_name in trip_context_lookup_map.items():
        if col in df.columns:
            decode_column_with_lookup_inplace(df, col, lookups[table_name], keep_original=False)

        # 7) Coordenadas WGS84 + H3, pero en columnas no canónicas.
    add_wgs84_from_xy(
        df,
        x_col="OrigenCoordX",
        y_col="OrigenCoordY",
        out_lon="OrigenCoordLon",
        out_lat="OrigenCoordLat",
        source_crs=source_crs,
        target_crs=target_crs,
    )
    add_wgs84_from_xy(
        df,
        x_col="DestinoCoordX",
        y_col="DestinoCoordY",
        out_lon="DestinoCoordLon",
        out_lat="DestinoCoordLat",
        source_crs=source_crs,
        target_crs=target_crs,
    )

    sort_cols = [c for c in ["Viaje", "Etapa"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols, kind="stable").copy()

    df["NumeroEtapa"] = df.groupby("Viaje").cumcount() + 1

    # Propago peso del viaje con nombre no canónico.
    trip_factor_cols_present = [
        f"trip_{c}" for c in TRIP_FACTOR_COLS if f"trip_{c}" in df.columns
    ]
    if trip_factor_cols_present:
        df["factor_expansion"] = first_non_null_numeric(df, trip_factor_cols_present)

    if "trip_Etapas" in df.columns:
        df["cantidad_etapas"] = pd.to_numeric(df["trip_Etapas"], errors="coerce").astype("Int64")


    return df

In [97]:
eod_viajes = pd.read_csv(DATA_PATH / "Viajes.csv", sep=";", dtype=str, low_memory=False)
eod_personas = pd.read_csv(DATA_PATH / "Personas.csv", sep=";", dtype=str, low_memory=False)
eod_hogares = pd.read_csv(DATA_PATH / "Hogares.csv", sep=";", dtype=str, low_memory=False)


In [98]:
df_eod_trips_for_import = prepare_eod_triplevel_for_import(
    df_viajes=eod_viajes,
    df_personas=eod_personas,
    df_hogares=eod_hogares,
    aux_dir=DATA_PATH / "Tablas_parametros",
)

display(df_eod_trips_for_import)
display(df_eod_trips_for_import.columns)

,Hogar,Persona,Viaje,Etapas,ComunaOrigen,ComunaDestino,SectorOrigen,SectorDestino,ZonaOrigen,ZonaDestino,...,NumBicAdulto,NumBicNino,IngresoHogar,Comuna,OrigenCoordLon,OrigenCoordLat,DestinoCoordLon,DestinoCoordLat,factor_expansion,cantidad_etapas
0,173431,17343102,1734310202,1,MAIPÚ,MAIPÚ,Poniente,Poniente,400,407,...,0,0,789356,MAIPU,-70.774666,-33.531422,-70.735153,-33.495874,1.000000,1
1,173441,17344101,1734410101,2,MAIPÚ,LAS CONDES,Poniente,Oriente,407,307,...,0,0,633883,MAIPU,-70.738205,-33.500007,-70.567232,-33.408776,1.127220,2
2,173441,17344101,1734410102,2,LAS CONDES,MAIPÚ,Oriente,Poniente,307,407,...,0,0,633883,MAIPU,-70.567232,-33.408776,-70.738205,-33.500007,1.127220,2
3,173441,17344103,1734410301,2,MAIPÚ,ÑUÑOA,Poniente,Oriente,407,437,...,0,0,633883,MAIPU,-70.738205,-33.500007,-70.604904,-33.454153,1.127220,2
4,173441,17344103,1734410302,2,ÑUÑOA,MAIPÚ,Oriente,Poniente,437,407,...,0,0,633883,MAIPU,-70.604904,-33.454153,-70.738205,-33.500007,1.052764,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113586,743813,74381301,7438130102,1,LA GRANJA,LA GRANJA,Sur,Sur,235,244,...,0,0,318005,LA GRANJA,-70.619877,-33.515990,-70.621567,-33.521961,1.482104,1
113587,743813,74381302,7438130201,1,LA GRANJA,LA GRANJA,Sur,Sur,244,235,...,0,0,318005,LA GRANJA,-70.621567,-33.521961,-70.619877,-33.515990,1.482104,1
113588,743813,74381302,7438130202,1,LA GRANJA,LA GRANJA,Sur,Sur,235,244,...,0,0,318005,LA GRANJA,-70.619877,-33.515990,-70.621567,-33.521961,1.482104,1
113589,743823,74382301,7438230101,1,LA GRANJA,LA GRANJA,Sur,Sur,235,235,...,1,0,140007,LA GRANJA,NaN,NaN,NaN,NaN,1.143880,1


Index(['Hogar', 'Persona', 'Viaje', 'Etapas', 'ComunaOrigen', 'ComunaDestino',
       'SectorOrigen', 'SectorDestino', 'ZonaOrigen', 'ZonaDestino',
       'Proposito', 'PropositoAgregado', 'ActividadDestino', 'MediosUsados',
       'ModoAgregado', 'ModoPriPub', 'ModoMotor', 'HoraIni', 'HoraFin',
       'HoraMedia', 'TiempoViaje', 'TiempoMedio', 'Periodo', 'MinutosDespues',
       'CuadrasDespues', 'FactorLaboralNormal', 'FactorSabadoNormal',
       'FactorDomingoNormal', 'FactorLaboralEstival',
       'FactorFindesemanaEstival', 'CodigoTiempo', 'AnoNac', 'Sexo',
       'Actividad', 'Ocupacion', 'LicenciaConducir', 'PaseEscolar',
       'AdultoMayor', 'TieneIngresos', 'TramoIngreso', 'Fecha', 'TipoDia',
       'Temporada', 'NumVeh', 'NumBicAdulto', 'NumBicNino', 'IngresoHogar',
       'Comuna', 'OrigenCoordLon', 'OrigenCoordLat', 'DestinoCoordLon',
       'DestinoCoordLat', 'factor_expansion', 'cantidad_etapas'],
      dtype='object')

In [ ]:
eod_etapas = pd.read_csv(DATA_PATH / "Etapas.csv", sep=";", dtype=str, low_memory=False, encoding="latin-1")
display(eod_etapas.columns)
print("cantidad de columnas:", len(eod_etapas.columns))

Index(['Hogar', 'Persona', 'Viaje', 'Etapa', 'ZonaOrigen', 'ZonaDestino',
       'ComunaOrigen', 'ComunaDestino', 'OrigenCoordX', 'OrigenCoordY',
       'DestinoCoordX', 'DestinoCoordY', 'Modo', 'CuadrasAntes',
       'MinutosAntes', 'Autopistas', 'NoUsaAutopistas', 'Estaciona',
       'CostoEstacionamiento', 'FormaPago', 'EstacionTrenIni',
       'EstacionTrenFin', 'TarifaTren', 'RecorridoTransantiago',
       'TiempoEsperaTstgo', 'TiempoEsperaBus', 'BusesPerdidos',
       'TarifaBusNoTransantiago', 'EstacionMetroIni', 'EstacionMetroFin',
       'HorarioMetro', 'MetrosPerdidos', 'EstacionMetroCambio', 'RecorridoTxc',
       'TiempoEsperaTxc', 'TarifaTxc', 'TiempoEsperaTaxi', 'TarifaTaxi',
       'PropiedadBicicleta', 'UsaCiclovia', 'CirculacionBicicleta',
       'EstacionaBicicleta', 'ModoEstacionaBicicleta', 'UsoHabitualBicicleta'],
      dtype='object')

cantidad de columnas: 44


In [100]:
df_eod_stages_for_import = prepare_eod_stagelevel_for_import(
    df_etapas=eod_etapas,
    df_viajes=eod_viajes,        # opcional, pero recomendado para traer propósito/weights/contexto
    df_personas=eod_personas,
    df_hogares=eod_hogares,
    aux_dir=DATA_PATH / "Tablas_parametros",
)
display(df_eod_stages_for_import)

,Hogar,Persona,Viaje,Etapa,ZonaOrigen,ZonaDestino,ComunaOrigen,ComunaDestino,Modo,CuadrasAntes,...,trip_FactorDomingoNormal,trip_FactorLaboralEstival,trip_FactorFindesemanaEstival,OrigenCoordLon,OrigenCoordLat,DestinoCoordLon,DestinoCoordLat,NumeroEtapa,factor_expansion,cantidad_etapas
0,100010,10001001,1000100101,10001001011,786,786,BUIN,BUIN,Enteramente a pie,0,...,"1,48210391",NaN,NaN,-70.779034,-33.729444,-70.778858,-33.729996,1,1.482104,1
1,100010,10001001,1000100102,10001001021,786,786,BUIN,BUIN,Enteramente a pie,0,...,"1,48210391",NaN,NaN,-70.778858,-33.729996,-70.779034,-33.729444,1,1.482104,1
2,100010,10001002,1000100201,10001002011,786,786,BUIN,BUIN,Enteramente a pie,0,...,"1,48210391",NaN,NaN,-70.779034,-33.729444,-70.776804,-33.730774,1,1.482104,1
3,100010,10001002,1000100202,10001002021,786,786,BUIN,BUIN,Enteramente a pie,0,...,"1,48210391",NaN,NaN,-70.776804,-33.730774,-70.776589,-33.730448,1,1.482104,1
4,100010,10001002,1000100203,10001002031,786,786,BUIN,BUIN,Enteramente a pie,0,...,"1,48210391",NaN,NaN,-70.776589,-33.730448,-70.779034,-33.729444,1,1.482104,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133439,743813,74381301,7438130102,74381301021,235,244,LA GRANJA,LA GRANJA,Enteramente a pie,0,...,"1,48210391",NaN,NaN,-70.619877,-33.515990,-70.621567,-33.521961,1,1.482104,1
133440,743813,74381302,7438130201,74381302011,244,235,LA GRANJA,LA GRANJA,Enteramente a pie,0,...,"1,48210391",NaN,NaN,-70.621567,-33.521961,-70.619877,-33.515990,1,1.482104,1
133441,743813,74381302,7438130202,74381302021,235,244,LA GRANJA,LA GRANJA,Enteramente a pie,0,...,"1,48210391",NaN,NaN,-70.619877,-33.515990,-70.621567,-33.521961,1,1.482104,1
133442,743823,74382301,7438230101,74382301011,235,235,LA GRANJA,LA GRANJA,Bicicleta,0,...,"1,14387994",NaN,NaN,-70.622007,-33.522622,-70.619877,-33.515990,1,1.143880,1


In [ ]:
#df_eod_trips_for_import.head(1000).to_csv(DATA_PATH / "temp" / "preproccesed_trips.csv", sep=";", index=False)
#df_eod_stages_for_import.head(1000).to_csv(DATA_PATH / "temp" / "preproccesed_stages.csv", sep=";", index=False)

## Prueba de import trips

In [102]:
from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec, TripSchemaEffective
from pylondrina.datasets import TripDataset
from pylondrina.reports import ImportReport, Issue
from pylondrina.errors import ImportError as PylondrinaImportError, SchemaError

In [103]:
EOD_TRIPS_IMPORT_SCHEMA = TripSchema(
    version="1.1",
    fields={
        # required core de movements
        "movement_id": FieldSpec(
            "movement_id",
            "string",
            required=True,
            constraints={"nullable": False, "unique": True},
        ),
        "user_id": FieldSpec(
            "user_id",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "origin_longitude": FieldSpec(
            "origin_longitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -180.0, "max": 180.0}},
        ),
        "origin_latitude": FieldSpec(
            "origin_latitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -90.0, "max": 90.0}},
        ),
        "destination_longitude": FieldSpec(
            "destination_longitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -180.0, "max": 180.0}},
        ),
        "destination_latitude": FieldSpec(
            "destination_latitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -90.0, "max": 90.0}},
        ),
        "origin_h3_index": FieldSpec(
            "origin_h3_index",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "destination_h3_index": FieldSpec(
            "destination_h3_index",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "trip_id": FieldSpec(
            "trip_id",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "movement_seq": FieldSpec(
            "movement_seq",
            "int",
            required=True,
            constraints={"nullable": False},
        ),

        # tiempos base / opcionales
        "origin_time_utc": FieldSpec("origin_time_utc", "datetime", required=False),
        "destination_time_utc": FieldSpec("destination_time_utc", "datetime", required=False),
        "origin_time_local_hhmm": FieldSpec("origin_time_local_hhmm", "string", required=False),
        "destination_time_local_hhmm": FieldSpec("destination_time_local_hhmm", "string", required=False),

        # base no categóricos
        "origin_municipality": FieldSpec("origin_municipality", "string", required=False),
        "destination_municipality": FieldSpec("destination_municipality", "string", required=False),
        "trip_weight": FieldSpec("trip_weight", "float", required=False),
        "mode_sequence": FieldSpec("mode_sequence", "string", required=False),

        # base categóricos
        "mode": FieldSpec(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[
                    "walk", "bicycle", "scooter", "motorcycle", "car",
                    "taxi", "ride_hailing", "bus", "metro", "train", "other"
                ],
                extendable=True,
            ),
        ),
        "purpose": FieldSpec(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[
                    "home", "work", "education", "shopping",
                    "errand", "health", "leisure", "transfer", "other"
                ],
                extendable=True,
            ),
        ),
        "day_type": FieldSpec(
            "day_type",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["weekday", "weekend", "holiday"],
                extendable=True,
            ),
        ),
        "time_period": FieldSpec(
            "time_period",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["night", "morning", "midday", "afternoon", "evening"],
                extendable=True,
            ),
        ),
        "user_gender": FieldSpec(
            "user_gender",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["female", "male", "other", "unknown"],
                extendable=True,
            ),
        ),
        "user_age_group": FieldSpec(
            "user_age_group",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["0-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65-plus", "unknown"],
                extendable=True,
            ),
        ),
        "income_quintile": FieldSpec(
            "income_quintile",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["1", "2", "3", "4", "5", "unknown"],
                extendable=True,
            ),
        ),
    },
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
    ],
)

In [104]:
EOD_TRIPS_IMPORT_OPTIONS = ImportOptions(
    keep_extra_fields=True,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone=None,
)

In [105]:
EOD_TRIPS_FIELD_CORRESPONDENCE = {
    "user_id": "Persona",
    "movement_id": "Viaje",
    # trip_id se deriva desde movement_id por single_stage=True
    # movement_seq se fija en 0 por single_stage=True

    "origin_longitude": "OrigenCoordLon",
    "origin_latitude": "OrigenCoordLat",
    "destination_longitude": "DestinoCoordLon",
    "destination_latitude": "DestinoCoordLat",

    # H3 no viene en preprocess: import debe derivarlo
    # origin_h3_index / destination_h3_index se materializan desde coords + h3_resolution

    "origin_time_local_hhmm": "HoraIni",
    "destination_time_local_hhmm": "HoraFin",

    "origin_municipality": "ComunaOrigen",
    "destination_municipality": "ComunaDestino",

    "mode": "ModoAgregado",
    "purpose": "Proposito",
    "day_type": "TipoDia",
    "user_gender": "Sexo",

    "trip_weight": "factor_expansion",
}

In [106]:
EOD_TRIPS_VALUE_CORRESPONDENCE = {
    "mode": {
        "Auto": "car",
        "Bus TS": "bus",
        "Bus no TS": "bus",
        "Metro": "metro",
        "Taxi Colectivo": "taxi",
        "Taxi": "taxi",
        "Caminata": "walk",
        "Bicicleta": "bicycle",

        # extensiones EOD agregadas, si aparecen
        "Bus TS - Bus no TS": "bus",
        "Auto - Metro": "other",
        "Bus TS - Metro": "other",
        "Bus no TS - Metro": "other",
        "Taxi Colectivo - Metro": "other",
        "Taxi - Metro": "other",
        "Otros - Metro": "other",
        "Otros - Bus TS": "other",
        "Otros - Bus TS - Metro": "other",
        "Otros": "other",
    },
    "purpose": {
        "volver a casa": "home",
        "Volver a casa": "home",
        "Al trabajo": "work",
        "Por trabajo": "work",
        "Al estudio": "education",
        "Por estudio": "education",
        "De compras": "shopping",
        "Buscar o Dejar a alguien": "errand",
        "Buscar o dejar algo": "errand",
        "Trámites": "errand",
        "De salud": "health",
        "Comer o Tomar algo": "leisure",
        "Visitar a alguien": "leisure",
        "Recreación": "leisure",
        "Otra actividad": "other",
    },
    "day_type": {
        "Laboral": "weekday",
        "Fin de Semana": "weekend",
    },
    "user_gender": {
        "Hombre": "male",
        "Mujer": "female",
    },
}

In [110]:
trips_eod_trips, report_eod_trips = import_trips_from_dataframe(
    df_eod_trips_for_import.head(1000),
    schema=EOD_TRIPS_IMPORT_SCHEMA,
    source_name="EOD trips level-1 manual preprocess",
    options=EOD_TRIPS_IMPORT_OPTIONS,
    field_correspondence=EOD_TRIPS_FIELD_CORRESPONDENCE,
    value_correspondence=EOD_TRIPS_VALUE_CORRESPONDENCE,
    provenance={
        "source": {
            "name": "EOD",
            "entity": "trips",
            "version": "EOD_STGO",
        },
        "notes": ["preprocess manual nivel 1 en notebook"],
    },
    h3_resolution=8,
)

In [115]:
display(trips_eod_trips.data.head(10))
display(report_eod_trips.issues)

,trip_id,Hogar,user_id,movement_id,Etapas,origin_municipality,destination_municipality,SectorOrigen,SectorDestino,ZonaOrigen,...,Comuna,origin_longitude,origin_latitude,destination_longitude,destination_latitude,trip_weight,cantidad_etapas,origin_h3_index,destination_h3_index,movement_seq
0,1734310202,173431,17343102,1734310202,1,MAIPÚ,MAIPÚ,Poniente,Poniente,400,...,MAIPU,-70.774666,-33.531422,-70.735153,-33.495874,1.000000,1,88b2c540d1fffff,88b2c5429bfffff,0
1,1734410101,173441,17344101,1734410101,2,MAIPÚ,LAS CONDES,Poniente,Oriente,407,...,MAIPU,-70.738205,-33.500007,-70.567232,-33.408776,1.127220,2,88b2c5429bfffff,88b2c519a9fffff,0
2,1734410102,173441,17344101,1734410102,2,LAS CONDES,MAIPÚ,Oriente,Poniente,307,...,MAIPU,-70.567232,-33.408776,-70.738205,-33.500007,1.127220,2,88b2c519a9fffff,88b2c5429bfffff,0
3,1734410301,173441,17344103,1734410301,2,MAIPÚ,ÑUÑOA,Poniente,Oriente,407,...,MAIPU,-70.738205,-33.500007,-70.604904,-33.454153,1.127220,2,88b2c5429bfffff,88b2c554dbfffff,0
4,1734410302,173441,17344103,1734410302,2,ÑUÑOA,MAIPÚ,Oriente,Poniente,437,...,MAIPU,-70.604904,-33.454153,-70.738205,-33.500007,1.052764,2,88b2c554dbfffff,88b2c5429bfffff,0
5,1734510101,173451,17345101,1734510101,1,MAIPÚ,SANTIAGO,Poniente,Centro,418,...,MAIPU,-70.786138,-33.556823,-70.655200,-33.440072,1.482104,1,88b2c540bbfffff,88b2c55411fffff,0
6,1734510102,173451,17345101,1734510102,1,SANTIAGO,MAIPÚ,Centro,Poniente,18,...,MAIPU,-70.655200,-33.440072,-70.786138,-33.556823,1.482104,1,88b2c55411fffff,88b2c540bbfffff,0
7,1734510103,173451,17345101,1734510103,1,MAIPÚ,MAIPÚ,Poniente,Poniente,418,...,MAIPU,NaN,NaN,NaN,NaN,1.482104,1,<NA>,<NA>,0
8,1734510104,173451,17345101,1734510104,1,MAIPÚ,MAIPÚ,Poniente,Poniente,418,...,MAIPU,NaN,NaN,NaN,NaN,1.482104,1,<NA>,<NA>,0
9,1734510201,173451,17345102,1734510201,1,MAIPÚ,MAIPÚ,Poniente,Poniente,418,...,MAIPU,NaN,NaN,NaN,NaN,1.482104,1,<NA>,<NA>,0


[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'origin_time_utc' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='origin_time_utc', source_field=None, row_count=None, details={'field': 'origin_time_utc', 'source_columns': ['Hogar', 'Persona', 'Viaje', 'Etapas', 'ComunaOrigen', 'ComunaDestino', 'SectorOrigen', 'SectorDestino', 'ZonaOrigen', 'ZonaDestino', 'Proposito', 'PropositoAgregado', 'ActividadDestino', 'MediosUsados', 'ModoAgregado', 'ModoPriPub', 'ModoMotor', 'HoraIni', 'HoraFin', 'HoraMedia', 'TiempoViaje', 'TiempoMedio', 'Periodo', 'MinutosDespues', 'CuadrasDespues', 'FactorLaboralNormal', 'FactorSabadoNormal', 'FactorDomingoNormal', 'FactorLaboralEstival', 'FactorFindesemanaEstival', 'CodigoTiempo', 'AnoNac', 'Sexo', 'Actividad', 'Ocupacion', 'LicenciaConducir', 'PaseEscolar', 'AdultoMayor', 'TieneIngresos', 'TramoIngreso', 'Fecha', 'TipoDia', 'Temporada', 'NumVeh', 'NumBicAdulto', 'NumBicNino', 'I

In [107]:
EOD_STAGES_IMPORT_SCHEMA = EOD_TRIPS_IMPORT_SCHEMA
EOD_STAGES_IMPORT_OPTIONS = ImportOptions(
    keep_extra_fields=True,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=False,
    source_timezone=None,
)

In [108]:
EOD_STAGES_FIELD_CORRESPONDENCE = {
    "user_id": "Persona",
    "trip_id": "Viaje",
    "movement_id": "Etapa",
    "movement_seq": "NumeroEtapa",

    "origin_longitude": "OrigenCoordLon",
    "origin_latitude": "OrigenCoordLat",
    "destination_longitude": "DestinoCoordLon",
    "destination_latitude": "DestinoCoordLat",

    # H3 se deriva en import

    "origin_municipality": "ComunaOrigen",
    "destination_municipality": "ComunaDestino",

    "mode": "Modo",
    "purpose": "trip_Proposito",
    "day_type": "TipoDia",
    "user_gender": "Sexo",

    "trip_weight": "factor_expansion",
}

In [109]:
EOD_STAGES_VALUE_CORRESPONDENCE = {
    "mode": {
        "Auto Chofer": "car",
        "Auto Acompañante": "car",
        "Bus alimentador": "bus",
        "Bus troncal": "bus",
        "Bus institucional": "bus",
        "Bus interurbano o rural": "bus",
        "Bus urbano con pago al conductor (Metrobus y otros)": "bus",
        "Metro": "metro",
        "Taxi colectivo": "taxi",
        "Taxi o radiotaxi": "taxi",
        "Enteramente a pie": "walk",
        "Bicicleta": "bicycle",
        "Motocicleta": "motorcycle",
        "Motocicleta Acompañante": "motorcycle",
        "Tren": "train",
        "Servicio Informal": "other",
        "Furgón escolar, como pasajero": "other",
        "Furgón escolar, como chofer o acompañante": "other",
    },
    "purpose": {
        "volver a casa": "home",
        "Volver a casa": "home",
        "Al trabajo": "work",
        "Por trabajo": "work",
        "Al estudio": "education",
        "Por estudio": "education",
        "De compras": "shopping",
        "Buscar o Dejar a alguien": "errand",
        "Buscar o dejar algo": "errand",
        "Trámites": "errand",
        "De salud": "health",
        "Comer o Tomar algo": "leisure",
        "Visitar a alguien": "leisure",
        "Recreación": "leisure",
        "Otra actividad": "other",
    },
    "day_type": {
        "Laboral": "weekday",
        "Fin de Semana": "weekend",
    },
    "user_gender": {
        "Hombre": "male",
        "Mujer": "female",
    },
}

In [116]:
trips_eod_stages, report_eod_stages = import_trips_from_dataframe(
    df_eod_stages_for_import.head(1000),
    schema=EOD_STAGES_IMPORT_SCHEMA,
    source_name="EOD stages level-1 manual preprocess",
    options=EOD_STAGES_IMPORT_OPTIONS,
    field_correspondence=EOD_STAGES_FIELD_CORRESPONDENCE,
    value_correspondence=EOD_STAGES_VALUE_CORRESPONDENCE,
    provenance={
        "source": {
            "name": "EOD",
            "entity": "stages",
            "version": "EOD_STGO",
        },
        "notes": ["preprocess manual nivel 1 en notebook"],
    },
    h3_resolution=8,
)

In [117]:
display(trips_eod_stages.data.head(10))
display(report_eod_stages.issues)

,Hogar,user_id,trip_id,movement_id,ZonaOrigen,ZonaDestino,origin_municipality,destination_municipality,mode,CuadrasAntes,...,trip_FactorFindesemanaEstival,origin_longitude,origin_latitude,destination_longitude,destination_latitude,movement_seq,trip_weight,cantidad_etapas,origin_h3_index,destination_h3_index
0,100010,10001001,1000100101,10001001011,786,786,BUIN,BUIN,walk,0,...,NaN,-70.779034,-33.729444,-70.778858,-33.729996,1,1.482104,1,88b2c5792bfffff,88b2c5793dfffff
1,100010,10001001,1000100102,10001001021,786,786,BUIN,BUIN,walk,0,...,NaN,-70.778858,-33.729996,-70.779034,-33.729444,1,1.482104,1,88b2c5793dfffff,88b2c5792bfffff
2,100010,10001002,1000100201,10001002011,786,786,BUIN,BUIN,walk,0,...,NaN,-70.779034,-33.729444,-70.776804,-33.730774,1,1.482104,1,88b2c5792bfffff,88b2c5793dfffff
3,100010,10001002,1000100202,10001002021,786,786,BUIN,BUIN,walk,0,...,NaN,-70.776804,-33.730774,-70.776589,-33.730448,1,1.482104,1,88b2c5793dfffff,88b2c5793dfffff
4,100010,10001002,1000100203,10001002031,786,786,BUIN,BUIN,walk,0,...,NaN,-70.776589,-33.730448,-70.779034,-33.729444,1,1.482104,1,88b2c5793dfffff,88b2c5792bfffff
5,100010,10001002,1000100204,10001002041,786,786,BUIN,BUIN,walk,0,...,NaN,-70.779034,-33.729444,-70.781813,-33.732772,1,1.482104,1,88b2c5792bfffff,88b2c57923fffff
6,100010,10001002,1000100205,10001002051,786,786,BUIN,BUIN,walk,0,...,NaN,-70.781813,-33.732772,-70.779034,-33.729444,1,1.482104,1,88b2c57923fffff,88b2c5792bfffff
7,100020,10002001,1000200101,10002001011,785,786,BUIN,BUIN,walk,0,...,NaN,-70.744339,-33.737279,-70.736059,-33.732450,1,1.482104,1,88b2c57913fffff,88b2c579cdfffff
8,100020,10002001,1000200102,10002001021,786,785,BUIN,BUIN,walk,0,...,NaN,-70.736059,-33.732450,-70.744691,-33.743695,1,1.482104,1,88b2c579cdfffff,88b2c579e9fffff
9,100020,10002001,1000200103,10002001031,785,785,BUIN,BUIN,walk,0,...,NaN,NaN,NaN,NaN,NaN,1,1.482104,1,<NA>,<NA>


[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'origin_time_utc' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='origin_time_utc', source_field=None, row_count=None, details={'field': 'origin_time_utc', 'source_columns': ['Hogar', 'Persona', 'Viaje', 'Etapa', 'ZonaOrigen', 'ZonaDestino', 'ComunaOrigen', 'ComunaDestino', 'Modo', 'CuadrasAntes', 'MinutosAntes', 'Autopistas', 'NoUsaAutopistas', 'Estaciona', 'CostoEstacionamiento', 'FormaPago', 'EstacionTrenIni', 'EstacionTrenFin', 'TarifaTren', 'RecorridoTransantiago', 'TiempoEsperaTstgo', 'TiempoEsperaBus', 'BusesPerdidos', 'TarifaBusNoTransantiago', 'EstacionMetroIni', 'EstacionMetroFin', 'HorarioMetro', 'MetrosPerdidos', 'EstacionMetroCambio', 'RecorridoTxc', 'TiempoEsperaTxc', 'TarifaTxc', 'TiempoEsperaTaxi', 'TarifaTaxi', 'PropiedadBicicleta', 'UsaCiclovia', 'CirculacionBicicleta', 'EstacionaBicicleta', 'ModoEstacionaBicicleta', 'UsoHabitualBicicleta', '

# Nivel 3

## Para EOD - Trips / viajes

### 1) Uso normal, rápido, sin decisiones de schema/campos/valores en la factory

In [3]:
import pandas as pd

from pylondrina.sources.helpers import import_trips_from_profile
from scripts.source_profiles.factories_eod.make_eod_trips_profile import (
    make_eod_trips_profile,
    EOD_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
)

eod_viajes = pd.read_csv(DATA_PATH / "Viajes.csv", sep=";", dtype=str, low_memory=False)
eod_personas = pd.read_csv(DATA_PATH / "Personas.csv", sep=";", dtype=str, low_memory=False)
eod_hogares = pd.read_csv(DATA_PATH / "Hogares.csv", sep=";", dtype=str, low_memory=False)

profile = make_eod_trips_profile(
    aux_dir=DATA_PATH / "Tablas_parametros",
    df_personas=eod_personas,
    df_hogares=eod_hogares,
    attach_person_fields=True,
    attach_household_fields=False,
)

# Convención de inspección clara
print("Inspeccion de trips:")
display(profile.schema_override)
display(profile.default_options)
display(profile.default_field_correspondence)
display(profile.default_value_correspondence)

Inspeccion de trips:


{'version': '1.1',
 'required': ['movement_id',
              'user_id',
              'origin_longitude',
              'origin_latitude',
              'destination_longitude',
              'destination_latitude',
              'origin_h3_index',
              'destination_h3_index',
              'trip_id',
              'movement_seq'],
 'semantic_rules': None,
 'fields': {'movement_id': {'name': 'movement_id',
                            'dtype': 'string',
                            'required': True,
                            'constraints': {'nullable': False, 'unique': True},
                            'domain': None},
            'user_id': {'name': 'user_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'origin_longitude': {'name': 'origin_longitude',
                                 'dtype': 'float',
                                 'required': True,
                                 'constraints': {'nullable': False, 'range': {'min': -180.0, 'max': 180.0}},
                                 'domain': None},
            'origin_latitude': {'name': 'origin_latitude',
                                'dtype': 'float',
                                'required': True,
                                'constraints': {'nullable': False, 'range': {'min': -90.0, 'max': 90.0}},
                                'domain': None},
            'destination_longitude': {'name': 'destination_longitude',
                                      'dtype': 'float',
                                      'required': True,
                                      'constraints': {'nullable': False, 'range': {'min': -180.0, 'max': 180.0}},
                                      'domain': None},
            'destination_latitude': {'name': 'destination_latitude',
                                     'dtype': 'float',
                                     'required': True,
                                     'constraints': {'nullable': False, 'range': {'min': -90.0, 'max': 90.0}},
                                     'domain': None},
            'origin_h3_index': {'name': 'origin_h3_index',
                                'dtype': 'string',
                                'required': True,
                                'constraints': {'nullable': False},
                                'domain': None},
            'destination_h3_index': {'name': 'destination_h3_index',
                                     'dtype': 'string',
                                     'required': True,
                                     'constraints': {'nullable': False},
                                     'domain': None},
            'trip_id': {'name': 'trip_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'movement_seq': {'name': 'movement_seq',
                             'dtype': 'int',
                             'required': True,
                             'constraints': {'nullable': False},
                             'domain': None},
            'origin_time_utc': {'name': 'origin_time_utc',
                                'dtype': 'datetime',
                                'required': False,
                                'constraints': None,
                                'domain': None},
            'destination_time_utc': {'name': 'destination_time_utc',
                                     'dtype': 'datetime',
                                     'required': False,
                                     'constraints': None,
                                     'domain': None},
            'origin_time_local_hhmm': {'name': 'origin_time_local_hhmm',
                                       'dtype': 'string',
                                       'r

ImportOptions(keep_extra_fields=True, selected_fields=None, strict=False, strict_domains=False, single_stage=True, source_timezone=None)

{'user_id': 'Persona',
 'movement_id': 'Viaje',
 'origin_longitude': 'OrigenCoordLon',
 'origin_latitude': 'OrigenCoordLat',
 'destination_longitude': 'DestinoCoordLon',
 'destination_latitude': 'DestinoCoordLat',
 'origin_time_local_hhmm': 'HoraIni',
 'destination_time_local_hhmm': 'HoraFin',
 'origin_municipality': 'ComunaOrigen',
 'destination_municipality': 'ComunaDestino',
 'mode': 'ModoAgregado',
 'purpose': 'Proposito',
 'day_type': 'TipoDia',
 'user_gender': 'Sexo',
 'trip_weight': 'factor_expansion'}

{'mode': {'Auto': 'car',
  'Bus TS': 'bus',
  'Bus no TS': 'bus',
  'Metro': 'metro',
  'Taxi Colectivo': 'taxi',
  'Taxi': 'taxi',
  'Caminata': 'walk',
  'Bicicleta': 'bicycle',
  'Bus TS - Bus no TS': 'bus',
  'Auto - Metro': 'other',
  'Bus TS - Metro': 'other',
  'Bus no TS - Metro': 'other',
  'Taxi Colectivo - Metro': 'other',
  'Taxi - Metro': 'other',
  'Otros - Metro': 'other',
  'Otros - Bus TS': 'other',
  'Otros - Bus TS - Metro': 'other',
  'Otros': 'other'},
 'purpose': {'volver a casa': 'home',
  'Volver a casa': 'home',
  'Al trabajo': 'work',
  'Por trabajo': 'work',
  'Al estudio': 'education',
  'Por estudio': 'education',
  'De compras': 'shopping',
  'Buscar o Dejar a alguien': 'errand',
  'Buscar o dejar algo': 'errand',
  'Trámites': 'errand',
  'De salud': 'health',
  'Comer o Tomar algo': 'leisure',
  'Visitar a alguien': 'leisure',
  'Recreación': 'leisure',
  'Otra actividad': 'other'},
 'day_type': {'Laboral': 'weekday', 'Fin de Semana': 'weekend'},
 'user_

In [4]:
trips_eod, report_eod = import_trips_from_profile(
    profile=profile,
    df=eod_viajes.head(5000),
    source_name="EOD trips level-3 factory",
    provenance=EOD_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)
display(trips_eod.data.head())
display(report_eod.issues)

,trip_id,Hogar,user_id,movement_id,Etapas,origin_municipality,destination_municipality,SectorOrigen,SectorDestino,ZonaOrigen,ZonaDestino,purpose,PropositoAgregado,ActividadDestino,MediosUsados,mode,ModoPriPub,ModoMotor,origin_time_local_hhmm,destination_time_local_hhmm,HoraMedia,TiempoViaje,TiempoMedio,Periodo,MinutosDespues,CuadrasDespues,FactorLaboralNormal,FactorSabadoNormal,FactorDomingoNormal,FactorLaboralEstival,FactorFindesemanaEstival,CodigoTiempo,AnoNac,user_gender,Actividad,Ocupacion,LicenciaConducir,PaseEscolar,AdultoMayor,TieneIngresos,TramoIngreso,mode_sequence,origin_longitude,origin_latitude,destination_longitude,destination_latitude,trip_weight,cantidad_etapas,origin_h3_index,destination_h3_index,movement_seq
0,1734310202,173431,17343102,1734310202,1,MAIPÚ,MAIPÚ,Poniente,Poniente,400,407,home,Otro,NaN,Bus alimentador,bus,Público,Motorizado,22:30,23:40,23:05,70,entre 1 hora y 1 hora 30 minutos,Noche (23:01 - 06:00),6,1,"1,00000000",NaN,NaN,NaN,NaN,Tiempo de viaje correcto,1972,female,Desempleado,NaN,No tiene licencia de conducir,No,No,no tiene ingresos,0,Bus alimentador,-70.774666,-33.531422,-70.735153,-33.495874,1.000000,1,88b2c540d1fffff,88b2c5429bfffff,0
1,1734410101,173441,17344101,1734410101,2,MAIPÚ,LAS CONDES,Poniente,Oriente,407,307,work,Trabajo,Servicios,Bus alimentador;Metro,other,Público,Motorizado,13:00,14:45,13:53,105,entre 1 hora 30 minutos y 2 horas,"Fuera de Punta 2 (9:01 - 10:00, 12:01 - 17:30, 20:31 - 23:00)",5,1,"1,12721985",NaN,NaN,NaN,NaN,Tiempo de viaje correcto,1991,male,Trabaja,Empleado u obrero del sector privado,No tiene licencia de conducir,No,No,"Sí, tiene ingresos",Entre 200.001 y 400.000 pesos,Bus alimentador;Metro,-70.738205,-33.500007,-70.567232,-33.408776,1.127220,2,88b2c5429bfffff,88b2c519a9fffff,0
2,1734410102,173441,17344101,1734410102,2,LAS CONDES,MAIPÚ,Oriente,Poniente,307,407,home,Trabajo,NaN,Metro;Bus alimentador,other,Público,Motorizado,22:00,23:30,22:45,90,entre 1 hora y 1 hora 30 minutos,"Fuera de Punta 2 (9:01 - 10:00, 12:01 - 17:30, 20:31 - 23:00)",10,2,"1,12721985",NaN,NaN,NaN,NaN,Tiempo de viaje correcto,1991,male,Trabaja,Empleado u obrero del sector privado,No tiene licencia de conducir,No,No,"Sí, tiene ingresos",Entre 200.001 y 400.000 pesos,Metro;Bus alimentador,-70.567232,-33.408776,-70.738205,-33.500007,1.127220,2,88b2c519a9fffff,88b2c5429bfffff,0
3,1734410301,173441,17344103,1734410301,2,MAIPÚ,ÑUÑOA,Poniente,Oriente,407,437,work,Trabajo,Servicios,Bus alimentador;Metro,other,Público,Motorizado,<NA>,<NA>,9:27,55,entre 31 minutos y 1 hora,"Fuera de Punta 2 (9:01 - 10:00, 12:01 - 17:30, 20:31 - 23:00)",10,2,"1,12721985",NaN,NaN,NaN,NaN,Tiempo de viaje correcto,1964,female,Trabaja,Empleado u obrero del sector privado,No tiene licencia de conducir,No,No,"Sí, tiene ingresos",No contesta,Bus alimentador;Metro,-70.738205,-33.500007,-70.604904,-33.454153,1.127220,2,88b2c5429bfffff,88b2c554dbfffff,0
4,1734410302,173441,17344103,1734410302,2,ÑUÑOA,MAIPÚ,Oriente,Poniente,437,407,home,Trabajo,NaN,Metro;Bus alimentador,other,Público,Motorizado,19:00,21:30,20:15,150,entre 2 horas y 2 horas 30 minutos,Punta Tarde (17:31 - 20:30),10,2,"1,05276356",NaN,NaN,NaN,NaN,Tiempo de viaje correcto,1964,female,Trabaja,Empleado u obrero del sector privado,No tiene licencia de conducir,No,No,"Sí, tiene ingresos",No contesta,Metro;Bus alimentador,-70.604904,-33.454153,-70.738205,-33.500007,1.052764,2,88b2c554dbfffff,88b2c5429bfffff,0


[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'day_type' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='day_type', source_field=None, row_count=None, details={'field': 'day_type', 'source_columns': ['Hogar', 'Persona', 'Viaje', 'Etapas', 'ComunaOrigen', 'ComunaDestino', 'SectorOrigen', 'SectorDestino', 'ZonaOrigen', 'ZonaDestino', 'Proposito', 'PropositoAgregado', 'ActividadDestino', 'MediosUsados', 'ModoAgregado', 'ModoPriPub', 'ModoMotor', 'HoraIni', 'HoraFin', 'HoraMedia', 'TiempoViaje', 'TiempoMedio', 'Periodo', 'MinutosDespues', 'CuadrasDespues', 'FactorLaboralNormal', 'FactorSabadoNormal', 'FactorDomingoNormal', 'FactorLaboralEstival', 'FactorFindesemanaEstival', 'CodigoTiempo', 'AnoNac', 'Sexo', 'Actividad', 'Ocupacion', 'LicenciaConducir', 'PaseEscolar', 'AdultoMayor', 'TieneIngresos', 'TramoIngreso', 'mode_sequence', 'OrigenCoordLon', 'OrigenCoordLat', 'DestinoCoordLon', 'DestinoCoordLat', 'fact

### 2) Uso más personalizado, con decisiones específicas

In [6]:
from pylondrina.sources.helpers import import_trips_from_profile
from scripts.source_profiles.factories_eod.make_eod_trips_profile import make_eod_trips_profile
from pylondrina.types import FieldCorrespondence, ValueCorrespondence
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.importing import ImportOptions

# -----------------------------------------------------------------------------
# Schema / options / correspondencias de ejemplo para uso más personalizado
# -----------------------------------------------------------------------------

EOD_TRIPS_CUSTOM_SCHEMA = TripSchema(
    version="1.1-eod-custom",
    fields={
        "movement_id": FieldSpec(
            "movement_id",
            "string",
            required=True,
            constraints={"nullable": False, "unique": True},
        ),
        "user_id": FieldSpec(
            "user_id",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "origin_longitude": FieldSpec(
            "origin_longitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -180.0, "max": 180.0}},
        ),
        "origin_latitude": FieldSpec(
            "origin_latitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -90.0, "max": 90.0}},
        ),
        "destination_longitude": FieldSpec(
            "destination_longitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -180.0, "max": 180.0}},
        ),
        "destination_latitude": FieldSpec(
            "destination_latitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -90.0, "max": 90.0}},
        ),
        "origin_h3_index": FieldSpec(
            "origin_h3_index",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "destination_h3_index": FieldSpec(
            "destination_h3_index",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "trip_id": FieldSpec(
            "trip_id",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "movement_seq": FieldSpec(
            "movement_seq",
            "int",
            required=True,
            constraints={"nullable": False},
        ),
        "origin_time_local_hhmm": FieldSpec("origin_time_local_hhmm", "string", required=False),
        "destination_time_local_hhmm": FieldSpec("destination_time_local_hhmm", "string", required=False),
        "origin_municipality": FieldSpec("origin_municipality", "string", required=False),
        "destination_municipality": FieldSpec("destination_municipality", "string", required=False),
        "mode": FieldSpec(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["walk", "bicycle", "car", "taxi", "bus", "metro", "train", "other"],
                extendable=True,
            ),
        ),
        "purpose": FieldSpec(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["home", "work", "education", "shopping", "errand", "health", "leisure", "other"],
                extendable=True,
            ),
        ),
        "day_type": FieldSpec(
            "day_type",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["weekday", "weekend", "holiday"],
                extendable=True,
            ),
        ),
        "user_gender": FieldSpec(
            "user_gender",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["female", "male", "other", "unknown"],
                extendable=True,
            ),
        ),
        "trip_weight": FieldSpec("trip_weight", "float", required=False),
    },
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
    ],
)

EOD_TRIPS_CUSTOM_OPTIONS = ImportOptions(
    keep_extra_fields=False,
    selected_fields=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
        "origin_time_local_hhmm",
        "destination_time_local_hhmm",
        "origin_municipality",
        "destination_municipality",
        "mode",
        "purpose",
        "day_type",
        "user_gender",
        "trip_weight",
    ],
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone=None,
)

# Ejemplo de override: usar propósito agregado en vez de propósito fino
EOD_TRIPS_CUSTOM_FIELD_CORRESPONDENCE: FieldCorrespondence = {
    "purpose": "PropositoAgregado",
}

EOD_TRIPS_CUSTOM_VALUE_CORRESPONDENCE: ValueCorrespondence = {
    "purpose": {
        "Trabajo y estudio": "other",
        "Otro" : "other",
        "Estudio": "education",
        "Trabajo": "work",
    },
    "mode": {
        "Tren": "train",
    },
}

EOD_TRIPS_CUSTOM_PROVENANCE_EXAMPLE = {
    "source": {
        "name": "EOD",
        "profile": "EOD_TRIPS_CUSTOM",
        "entity": "trips",
        "version": "EOD_STGO",
    },
    "notes": [
        "factory nivel 3 EOD con schema y correspondencias personalizadas",
        "purpose usa PropositoAgregado",
    ],
}

In [7]:
profile_custom = make_eod_trips_profile(
    aux_dir=DATA_PATH / "Tablas_parametros",
    df_personas=eod_personas,
    df_hogares=eod_hogares,
    attach_person_fields=True,
    attach_household_fields=True,
    schema_override=EOD_TRIPS_CUSTOM_SCHEMA,
    field_correspondence_override=EOD_TRIPS_CUSTOM_FIELD_CORRESPONDENCE,
    value_correspondence_override=EOD_TRIPS_CUSTOM_VALUE_CORRESPONDENCE,
    options_override={
        "keep_extra_fields": EOD_TRIPS_CUSTOM_OPTIONS.keep_extra_fields,
        "selected_fields": EOD_TRIPS_CUSTOM_OPTIONS.selected_fields,
        "strict": EOD_TRIPS_CUSTOM_OPTIONS.strict,
        "strict_domains": EOD_TRIPS_CUSTOM_OPTIONS.strict_domains,
        "single_stage": EOD_TRIPS_CUSTOM_OPTIONS.single_stage,
        "source_timezone": EOD_TRIPS_CUSTOM_OPTIONS.source_timezone,
    },
    profile_name="EOD_TRIPS_CUSTOM",
)

# Convención de inspección clara
print("Inspección:\n")
display(profile_custom.schema_override)
display(profile_custom.default_options)
display(profile_custom.default_field_correspondence)
display(profile_custom.default_value_correspondence)

Inspección:



{'version': '1.1-eod-custom',
 'required': ['movement_id',
              'user_id',
              'origin_longitude',
              'origin_latitude',
              'destination_longitude',
              'destination_latitude',
              'origin_h3_index',
              'destination_h3_index',
              'trip_id',
              'movement_seq'],
 'semantic_rules': None,
 'fields': {'movement_id': {'name': 'movement_id',
                            'dtype': 'string',
                            'required': True,
                            'constraints': {'nullable': False, 'unique': True},
                            'domain': None},
            'user_id': {'name': 'user_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'origin_longitude': {'name': 'origin_longitude',
                                 'dtype': 'float',
                                 'required': True,
                                 'constraints': {'nullable': False, 'range': {'min': -180.0, 'max': 180.0}},
                                 'domain': None},
            'origin_latitude': {'name': 'origin_latitude',
                                'dtype': 'float',
                                'required': True,
                                'constraints': {'nullable': False, 'range': {'min': -90.0, 'max': 90.0}},
                                'domain': None},
            'destination_longitude': {'name': 'destination_longitude',
                                      'dtype': 'float',
                                      'required': True,
                                      'constraints': {'nullable': False, 'range': {'min': -180.0, 'max': 180.0}},
                                      'domain': None},
            'destination_latitude': {'name': 'destination_latitude',
                                     'dtype': 'float',
                                     'required': True,
                                     'constraints': {'nullable': False, 'range': {'min': -90.0, 'max': 90.0}},
                                     'domain': None},
            'origin_h3_index': {'name': 'origin_h3_index',
                                'dtype': 'string',
                                'required': True,
                                'constraints': {'nullable': False},
                                'domain': None},
            'destination_h3_index': {'name': 'destination_h3_index',
                                     'dtype': 'string',
                                     'required': True,
                                     'constraints': {'nullable': False},
                                     'domain': None},
            'trip_id': {'name': 'trip_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'movement_seq': {'name': 'movement_seq',
                             'dtype': 'int',
                             'required': True,
                             'constraints': {'nullable': False},
                             'domain': None},
            'origin_time_local_hhmm': {'name': 'origin_time_local_hhmm',
                                       'dtype': 'string',
                                       'required': False,
                                       'constraints': None,
                                       'domain': None},
            'destination_time_local_hhmm': {'name': 'destination_time_local_hhmm',
                                            'dtype': 'string',
                                            'required': False,
                                            'constraints': None,
                                            'domain': None},
            'origin_municipality': {'name': 'origin_municipality',
              

ImportOptions(keep_extra_fields=False, selected_fields=['movement_id', 'user_id', 'origin_longitude', 'origin_latitude', 'destination_longitude', 'destination_latitude', 'origin_h3_index', 'destination_h3_index', 'trip_id', 'movement_seq', 'origin_time_local_hhmm', 'destination_time_local_hhmm', 'origin_municipality', 'destination_municipality', 'mode', 'purpose', 'day_type', 'user_gender', 'trip_weight'], strict=False, strict_domains=False, single_stage=True, source_timezone=None)

{'user_id': 'Persona',
 'movement_id': 'Viaje',
 'origin_longitude': 'OrigenCoordLon',
 'origin_latitude': 'OrigenCoordLat',
 'destination_longitude': 'DestinoCoordLon',
 'destination_latitude': 'DestinoCoordLat',
 'origin_time_local_hhmm': 'HoraIni',
 'destination_time_local_hhmm': 'HoraFin',
 'origin_municipality': 'ComunaOrigen',
 'destination_municipality': 'ComunaDestino',
 'mode': 'ModoAgregado',
 'purpose': 'PropositoAgregado',
 'day_type': 'TipoDia',
 'user_gender': 'Sexo',
 'trip_weight': 'factor_expansion'}

{'mode': {'Auto': 'car',
  'Bus TS': 'bus',
  'Bus no TS': 'bus',
  'Metro': 'metro',
  'Taxi Colectivo': 'taxi',
  'Taxi': 'taxi',
  'Caminata': 'walk',
  'Bicicleta': 'bicycle',
  'Bus TS - Bus no TS': 'bus',
  'Auto - Metro': 'other',
  'Bus TS - Metro': 'other',
  'Bus no TS - Metro': 'other',
  'Taxi Colectivo - Metro': 'other',
  'Taxi - Metro': 'other',
  'Otros - Metro': 'other',
  'Otros - Bus TS': 'other',
  'Otros - Bus TS - Metro': 'other',
  'Otros': 'other',
  'Tren': 'train'},
 'purpose': {'volver a casa': 'home',
  'Volver a casa': 'home',
  'Al trabajo': 'work',
  'Por trabajo': 'work',
  'Al estudio': 'education',
  'Por estudio': 'education',
  'De compras': 'shopping',
  'Buscar o Dejar a alguien': 'errand',
  'Buscar o dejar algo': 'errand',
  'Trámites': 'errand',
  'De salud': 'health',
  'Comer o Tomar algo': 'leisure',
  'Visitar a alguien': 'leisure',
  'Recreación': 'leisure',
  'Otra actividad': 'other',
  'Trabajo y estudio': 'other',
  'Otro': 'other',
  '

In [8]:

trips_eod_custom, report_eod_custom = import_trips_from_profile(
    profile=profile_custom,
    df=eod_viajes.head(5000),
    source_name="EOD trips level-3 custom factory",
    provenance=EOD_TRIPS_CUSTOM_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)


In [10]:
display(trips_eod_custom.data[1:20])
display(report_eod_custom.issues)

,trip_id,user_id,movement_id,origin_municipality,destination_municipality,purpose,mode,origin_time_local_hhmm,destination_time_local_hhmm,user_gender,day_type,origin_longitude,origin_latitude,destination_longitude,destination_latitude,trip_weight,origin_h3_index,destination_h3_index,movement_seq
1,1734410101,17344101,1734410101,MAIPÚ,LAS CONDES,work,other,13:00,14:45,male,weekday,-70.738205,-33.500007,-70.567232,-33.408776,1.127220,88b2c5429bfffff,88b2c519a9fffff,0
2,1734410102,17344101,1734410102,LAS CONDES,MAIPÚ,work,other,22:00,23:30,male,weekday,-70.567232,-33.408776,-70.738205,-33.500007,1.127220,88b2c519a9fffff,88b2c5429bfffff,0
3,1734410301,17344103,1734410301,MAIPÚ,ÑUÑOA,work,other,<NA>,<NA>,female,weekday,-70.738205,-33.500007,-70.604904,-33.454153,1.127220,88b2c5429bfffff,88b2c554dbfffff,0
4,1734410302,17344103,1734410302,ÑUÑOA,MAIPÚ,work,other,19:00,21:30,female,weekday,-70.604904,-33.454153,-70.738205,-33.500007,1.052764,88b2c554dbfffff,88b2c5429bfffff,0
5,1734510101,17345101,1734510101,MAIPÚ,SANTIAGO,other,car,10:00,11:00,female,weekend,-70.786138,-33.556823,-70.655200,-33.440072,1.482104,88b2c540bbfffff,88b2c55411fffff,0
6,1734510102,17345101,1734510102,SANTIAGO,MAIPÚ,other,car,15:00,15:45,female,weekend,-70.655200,-33.440072,-70.786138,-33.556823,1.482104,88b2c55411fffff,88b2c540bbfffff,0
7,1734510103,17345101,1734510103,MAIPÚ,MAIPÚ,home,walk,17:00,17:20,female,weekend,NaN,NaN,NaN,NaN,1.482104,<NA>,<NA>,0
8,1734510104,17345101,1734510104,MAIPÚ,MAIPÚ,home,walk,18:00,18:20,female,weekend,NaN,NaN,NaN,NaN,1.482104,<NA>,<NA>,0
9,1734510201,17345102,1734510201,MAIPÚ,MAIPÚ,other,walk,13:00,13:20,male,weekend,NaN,NaN,NaN,NaN,1.482104,<NA>,<NA>,0
10,1734510202,17345102,1734510202,MAIPÚ,MAIPÚ,other,walk,14:20,14:45,male,weekend,NaN,NaN,NaN,NaN,1.482104,<NA>,<NA>,0


[Issue(level='warning', code='IMP.TEMPORAL.TIER_LIMITED', message='El dataset fue importado con capacidad temporal limitada (tier_2); algunas operaciones posteriores quedarán restringidas.', field=None, source_field=None, row_count=None, details={'temporal_tier': 'tier_2', 'fields_present': ['origin_time_local_hhmm', 'destination_time_local_hhmm'], 'note': 'limited_temporal_capabilities'}),
 Issue(level='warning', code='IMP.TYPE.COERCE_PARTIAL', message="La conversión mínima del campo 'origin_time_local_hhmm' falló en algunas filas (26.6%); se marcarán como nulos para validación posterior.", field='origin_time_local_hhmm', source_field=None, row_count=None, details={'field': 'origin_time_local_hhmm', 'dtype_expected': 'string_hhmm', 'parse_fail_count': 1331, 'total_count': 5000, 'fail_rate': 0.2662, 'fallback': 'set_null', 'action': 'continue'}),
 Issue(level='warning', code='IMP.TYPE.COERCE_PARTIAL', message="La conversión mínima del campo 'origin_time_local_hhmm' falló en algunas fil